# Dual Aware Forward Selection with Batches and Tie-break

## L=10, A1 select = L*0.5, M=30

In [ ]:
import forward_walk_pipeline
from importlib import reload
reload(forward_walk_pipeline)
from forward_walk_pipeline import DualAwareForwardSelection
import json
from CPPMTO_instance_parser_v5 import parse_json_instance

#Change path

folder_paths = [
r"C:\Users\mkoseogl\Desktop\Geneva Data\relatota\instos_and_sols\instanto\nowito\Instance_1810_fix_euclid.json",
r"C:\Users\mkoseogl\Desktop\Geneva Data\relatota\instos_and_sols\instanto\nowito\Instance_1669_fix_euclid.json",
r"C:\Users\mkoseogl\Desktop\Geneva Data\relatota\instos_and_sols\instanto\nowito\Instance_4579_fix_euclid.json",
r"C:\Users\mkoseogl\Desktop\Geneva Data\relatota\instos_and_sols\instanto\nowito\Instance_5870_fix_euclid.json",
r"C:\Users\mkoseogl\Desktop\Geneva Data\relatota\instos_and_sols\instanto\nowito\Instance_19861_fix_euclid.json",
r"C:\Users\mkoseogl\Desktop\Geneva Data\relatota\instos_and_sols\instanto\nowito\Instance_17828_fix_euclid.json",
r"C:\Users\mkoseogl\Desktop\Geneva Data\relatota\instos_and_sols\instanto\nowito\Instance_38506_fix_euclid.json",
r"C:\Users\mkoseogl\Desktop\Geneva Data\relatota\instos_and_sols\instanto\nowito\Instance_40548_fix_euclid.json"
]


#Create save path 

save_path = r"C:\Users\mkoseogl\Desktop\Geneva Data\relatota\instos_and_sols\instanto\sols_heru\mr_greedy_forward_only_duals_L1005_fast"

# Main loop:
# - Parse Instance
# - Call heuristic pipeline to solve
# - Run heuristic pipeline
# - Save solution as json file
# - Update solution time for pure heuristic solve (initial selection + forward run with trials - initial model build) 
# Note: GUROBI MILP solution time does not include initial model building within solution time. 
# We include the following in our solution time: model copying, constraint additions to copy model, const additions to live model, full trial model build, full reoptimize model build times as it is part of the algorithm.

for path in folder_paths:
    parsed_data = parse_json_instance(path)
    # Assume you already have 'parsed_data', 'json_path', 'save_folder'
    print(f"\n[Running Instance]: {path}\n")
    pipe = DualAwareForwardSelection(parsed_data,
                                L=10, K_PT=2, mult_A1=0.5, H_PT=0, H_A1=10, theta=0.5,
                                P_bar=0.5, P_up=4.0, pt_price=2.0,
                                cost_bike=0.6, parking_fee=2.8, dep_cost=0,
                                tolerance=1e-3,
                                quiet=True, threads=4, time_limit_eval=None,
                                max_trial_add=3000, patience_add=3000)


    # D.1 batching routine: 
    # - Score: Disutility - present gamma_^{(o,d)} + reduced cost of capacity constraints for bike stations
    # - Sort based on score, pick a single arc. Check if bike station budget is satisfied with this arc.
    # - bike station budget is the remaining flow that can be added/removed from bike stations. 
    # - If station budget is violated, pair the arc with releaving arcs.
    # - Repeat until batch size is reached (<= M)


    # Trial Routine: 
    # - Add the arcs within the batch to the model copy
    # - Solve the model with incentivizing new flow variable in the obj function
    # - If obj value improves 

    best_val, elapsed = pipe.new_run(method="D1", M=30, mu_station=0.5, time_limit_eval=180)


    # Write JSON with your original structure and naming
    sol_path = pipe.write_json(json_file_path=path,
                            save_folder=save_path,
                            P_bar=0.5,
                            capacity=None,           # or capacity arg
                            inventory_level=None,    # or (low, high)
                            q_tol=0.0)

    with open(sol_path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    obj["Solution Time (s)"] = elapsed
    with open(sol_path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=4, ensure_ascii=False)

    print("Objective:", best_val, "Time(s):", elapsed, "Saved:", sol_path)

